# 11.2 - Agent Architectures: Reason, Plan, Reflect

**Module 11 - AI Agents** | Netsetos GenAI Engineering

This notebook builds the four canonical single-agent architectures - ReAct, Plan-and-Execute, ReWOO, and Reflexion - from scratch as tiny loops over one shared `search`/`calc` tool set, holding the model fixed so you can watch each wiring change the LLM-call count, cost, and failure mode. It then adds the fixed-code workflow patterns (routing) and a runnable decision ladder that picks the right shape for a task. The lesson's whole claim: after you pick a model, the architecture is the highest-leverage choice you make.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup

Install the one external dependency this lesson can call for real. Every architecture below runs on plain Python with scripted stand-ins for the model, so you can execute the whole notebook with no key at all - the only cell that needs the network is the live ReAct demo, which uses the Anthropic SDK.

In [ ]:
# Install Python dependencies used by this lesson (uncomment on Colab / fresh env).
# !pip install "anthropic>=0.40.0" python-dotenv -q  # noqa


**Explanation**

A dependency-install cell, not code that runs anything. The line is commented so it does nothing on a machine that already has the packages; uncomment it on Colab or a fresh environment.

**How the code works - step by step**
- **`anthropic>=0.40.0`** - the Anthropic Messages API client, used only by the live ReAct cell later.
- **`python-dotenv`** - lets you load keys from a `.env` file instead of hardcoding them.
- **`-q` / `# noqa`** - quiet install output; the noqa silences linting of the shell magic.

**In one line:** optional install - skip it if `anthropic` is already present.

**What you'll see:** (no output - environment prep)

## Environment: load your API key

Point the SDK at your key via the environment rather than pasting it into code. Any one provider key is enough to run the single live cell; the rest of the notebook needs none.

> **Needs an Anthropic API key** for the live ReAct cell only (set `ANTHROPIC_API_KEY`). Every other cell runs offline.

In [ ]:
import os
# Load API keys from the environment (or a .env file via python-dotenv).
# Never hardcode keys. Any ONE provider key is enough to start;
# the multi-provider demos are best with two or more.
os.environ.setdefault("ANTHROPIC_API_KEY", "")    # console.anthropic.com


**Explanation**

A configuration cell that seeds an environment variable. `setdefault` only fills the value if it is not already set, so a key you exported in your shell wins and is never overwritten by this blank default.

**How the code works - step by step**
- **`import os`** - gives access to the process environment.
- **`os.environ.setdefault("ANTHROPIC_API_KEY", "")`** - registers the key name with an empty fallback; the real key should come from your shell or a `.env` file, never from this string.

**In one line:** make the key available to the SDK without hardcoding it.

**What you'll see:** (no output - environment prep)

## 1 - The augmented LLM: the atom every architecture composes

Before any pattern, meet the building block: a model plus the tools it can call. Here that is a two-entry knowledge base with a `search` (retrieval) tool and a `calc` (action) tool. A blind model would guess the price and the GST; the augmented LLM can look one up and compute the other - and the architecture decides how many calls that takes.

In [ ]:
# THE AUGMENTED LLM - the atom every architecture composes: a model + tools (+ retrieval, memory).
# HOW you wire the calls around this atom - the architecture - moves cost and reliability
# more than swapping the model does. We script the model so the trace is deterministic.
KB = {"genai": "GenAI course: 14999 INR", "python": "Python course: 9999 INR"}
def search(q):  return KB.get(q.lower().split()[0], "not found")   # a retrieval tool
def calc(expr): return round(eval(expr, {"__builtins__": {}}), 2)  # an action tool
TOOLS = {"search": search, "calc": calc}

# A direct call would GUESS the price and the GST. The augmented LLM can look it up and compute it.
print("augmented LLM = model +", list(TOOLS))
print("search('genai') ->", search("genai"))
print("calc('14999*1.18') ->", calc("14999*1.18"))
print("same model, same tools - the architecture decides how many calls this takes")

# Output:
# augmented LLM = model + ['search', 'calc']
# search('genai') -> GenAI course: 14999 INR
# calc('14999*1.18') -> 17698.82
# same model, same tools - the architecture decides how many calls this takes

**Explanation**

A minimal reference harness that defines the shared tool set every later cell reuses - it is not a model call. The point it makes concretely: the same two tools can be wired many ways, and the wiring, not the model behind it, is the architecture.

**How the code works - step by step**
- **`KB`** - a two-item dict standing in for a retrieval store (GenAI and Python course prices).
- **`search(q)`** - looks up the first word of a query in `KB`; the retrieval tool.
- **`calc(expr)`** - evaluates arithmetic in a locked-down `eval` (no builtins); the action tool.
- **`TOOLS`** - the shared registry both tools live in, reused by every architecture below.
- The prints show a lookup and a computation working, then state the thesis.

**In one line:** model + tools = the atom; the count and shape of the calls around it = the architecture.

**What you'll see:** Prints the tool list `['search', 'calc']`, `search('genai')` returning the GenAI price string, `calc('14999*1.18')` returning `17698.82`, and a line reminding you the architecture decides the call count.

## 2 - ReAct: reason and act, interleaved

ReAct is the adaptive shape - Thought, Action, Observation, repeat - deciding the next step only after seeing the last result. It is the loop from 11.1 named as an architecture. Its strength is that every step is grounded in a real observation; its cost is one model call per step, so tokens and latency grow with the number of steps.

In [ ]:
# ReAct - reason & act, INTERLEAVED. One LLM call per step; the path is decided as it goes.
KB = {"genai": "GenAI course: 14999 INR", "python": "Python course: 9999 INR"}
def search(q):  return KB.get(q.lower().split()[0], "not found")
def calc(expr): return round(eval(expr, {"__builtins__": {}}), 2)
TOOLS = {"search": search, "calc": calc}
calls = {"n": 0}

def think(task, scratch):                          # SCRIPTED stand-in for one LLM call
    calls["n"] += 1
    used = [s for s in scratch if s.startswith("Act")]
    if not any("search" in u for u in used):
        return "Thought: I need the GenAI price first.", ("search", "genai")
    if not any("calc" in u for u in used):
        obs = used[0].split("->")[-1]                       # "GenAI course: 14999 INR"
        price = "".join(c for c in obs if c.isdigit())      # -> "14999"
        return "Thought: now add 18% GST.", ("calc", f"{price}*1.18")
    return "Thought: I have the answer.", ("finish", scratch[-1].split("->")[-1].strip())

def react(task, max_steps=5):
    scratch = []
    for step in range(max_steps):
        thought, (tool, arg) = think(task, scratch)     # REASON
        if tool == "finish":
            n_steps = len([s for s in scratch if s.startswith("Act")])
            print(f"  {thought}\n  Answer: {arg} INR ({n_steps} tool steps, {calls['n']} LLM calls)")
            return
        obs = TOOLS[tool](arg)                           # ACT
        scratch.append(thought)
        scratch.append(f"Act {tool}({arg}) -> {obs}")    # OBSERVE (fed back next step)
        print(f"  step {step+1}: {thought}  |  {tool}({arg}) -> {obs}")

react("What is the GenAI course price with 18% GST?")

# Output:
#   step 1: Thought: I need the GenAI price first.  |  search(genai) -> GenAI course: 14999 INR
#   step 2: Thought: now add 18% GST.  |  calc(14999*1.18) -> 17698.82
#   Thought: I have the answer.
#   Answer: 17698.82 INR (2 tool steps, 3 LLM calls)

**Explanation**

A from-scratch ReAct loop where a scripted `think` stands in for the per-step model call so the trace is deterministic. Read it as: reason to pick a tool, run it, feed the observation back, and let the next reasoning step depend on what was just seen.

**How the code works - step by step**
- **`think(task, scratch)`** - the stand-in for one LLM call; it inspects the scratchpad and picks the next action (search first, then calc, then finish), incrementing the call counter each time.
- **`react(task, max_steps)`** - the loop: call `think` to REASON, run the chosen tool to ACT, append the result to the scratchpad to OBSERVE, and stop at `finish` or the step cap.
- The GST calc in step 2 is built from the price observed in step 1 - that dependency is the adaptivity.
- **`max_steps=5`** - the iteration cap that prevents runaway loops.

**In one line:** one grounded reason-act-observe per step; call count grows with N.

**What you'll see:** Prints step 1 (search returns the GenAI price), step 2 (calc adds 18% GST -> 17698.82), then the final answer noting 2 tool steps but 3 LLM calls - one more call than tool steps because the final reasoning turn also counts.

## 3 - ReAct for real: the same loop on the Anthropic API

Now swap the scripted `think` for an actual model. This is 11.1's `stop_reason` loop against the Messages API: the model reasons in text, emits a `tool_use` block, you run the tool, append a `tool_result`, and repeat until it stops asking for tools. That interleaving of text reasoning and tool calls is ReAct.

> **Needs an Anthropic API key** (set `ANTHROPIC_API_KEY`).

In [ ]:
# ReAct FOR REAL - the same loop with the Anthropic Messages API (claude-sonnet-4-6).
import anthropic
client = anthropic.Anthropic()                       # reads ANTHROPIC_API_KEY
TOOLS = [{"name": "calc", "description": "Do arithmetic, e.g. 14999*1.18.",
          "input_schema": {"type": "object", "properties": {"expr": {"type": "string"}},
                           "required": ["expr"]}}]
def calc(expr): return round(eval(expr, {"__builtins__": {}}), 2)

messages = [{"role": "user", "content": "What is 14999 with 18% GST?"}]
max_steps = 5
for step in range(max_steps):                        # cap iterations - 11.1's reliability rule
    resp = client.messages.create(model="claude-sonnet-4-6", max_tokens=1024,
                                  tools=TOOLS, messages=messages)
    if resp.stop_reason != "tool_use":               # not tool_use -> the model reasoned to an answer
        print(resp.content[0].text); break
    messages.append({"role": "assistant", "content": resp.content})
    results = [{"type": "tool_result", "tool_use_id": b.id, "content": str(calc(**b.input))}
               for b in resp.content if b.type == "tool_use"]     # ACT
    messages.append({"role": "user", "content": results})         # OBSERVE

# Output: (illustrative minimal example - needs `pip install anthropic` + a key; in production add
#   a request timeout= and try/except.) The model REASONS in text between tool calls and ACTS via
#   tool_use - that interleaving IS ReAct. LangChain 1.0 gives you this for free:
#   from langchain.agents import create_agent; agent = create_agent(model, tools)  # runs the loop.

**Explanation**

The production version of cell 2 - a real ReAct loop using `claude-sonnet-4-6` with one declared `calc` tool. It shows how the same shape you hand-built maps onto the SDK's message/tool-result protocol, and notes that frameworks ship this loop prebuilt.

**How the code works - step by step**
- **`client` + `TOOLS`** - the Anthropic client (reads the env key) and a single typed `calc` tool schema.
- **The `for` loop** - capped at `max_steps`, exactly 11.1's reliability rule.
- **`resp.stop_reason != "tool_use"`** - the exit test: any other stop reason means the model reasoned its way to a final answer, which it prints.
- **On `tool_use`** - append the assistant turn, run `calc(**b.input)` for each tool call (ACT), and append the results as a `tool_result` user turn (OBSERVE).
- The closing comment notes `create_agent(model, tools)` in LangChain 1.0 runs this loop for you.

**In one line:** `tool_use` -> run tool -> append `tool_result` -> repeat; that IS ReAct.

**What you'll see:** With a valid key, prints the model's final answer for 14999 with 18% GST; the exact wording varies by run. Without a key or the `anthropic` package it raises rather than printing - it is an illustrative minimal example that omits the timeout and try/except you would add in production.

## 4 - Plan-and-Execute: plan first, then run

ReAct decides one step at a time; Plan-and-Execute decides them all at once. A capable model writes the full plan up front in a single call, then a cheap executor runs each step with no per-step reasoning - far fewer calls on the hot path, and a plan you can audit before anything runs. The catch: the planner commits before seeing any tool result, so a surprise mid-run forces a replan.

In [ ]:
# PLAN-AND-EXECUTE - a capable planner writes the whole plan FIRST; a cheap executor runs it.
KB = {"genai": "GenAI course: 14999 INR", "python": "Python course: 9999 INR"}
def search(q):  return KB.get(q.lower().split()[0], "not found")
def calc(expr): return round(eval(expr, {"__builtins__": {}}), 2)
TOOLS = {"search": search, "calc": calc}

def plan(task):                                   # ONE planner call, up front
    return [{"tool": "search", "arg": "genai"},
            {"tool": "search", "arg": "python"},
            {"tool": "calc",   "arg": "14999*1.18"},
            {"tool": "calc",   "arg": "9999*1.18"},
            {"tool": "answer", "arg": "compare the two GST-inclusive prices"}]

def execute(task):
    steps = plan(task)
    print(f"  planned {len(steps)} steps up front (1 planner call), now executing:")
    results = []
    for i, s in enumerate(steps, 1):
        if s["tool"] == "answer":
            print(f"    {i}. ANSWER: GenAI {results[2]} vs Python {results[3]} INR (with GST)")
            return
        out = TOOLS[s["tool"]](s["arg"])
        results.append(out)
        print(f"    {i}. {s['tool']}({s['arg']}) -> {out}")

execute("Compare GenAI and Python prices, both with 18% GST")

# If a step's result surprises the plan, we REPLAN (call the planner again). Demo:
def execute_with_replan(bad_step=2):
    steps = plan("x")
    for i, s in enumerate(steps, 1):
        if i == bad_step:
            print(f"  step {i} returned the unexpected -> REPLAN (1 extra planner call)")
            return
        print(f"  step {i}: {s['tool']}({s['arg']}) ok")
print("\n  brittle-plan demo:")
execute_with_replan()

# Output:
#   planned 5 steps up front (1 planner call), now executing:
#     1. search(genai) -> GenAI course: 14999 INR
#     2. search(python) -> Python course: 9999 INR
#     3. calc(14999*1.18) -> 17698.82
#     4. calc(9999*1.18) -> 11798.82
#     5. ANSWER: GenAI 17698.82 vs Python 11798.82 INR (with GST)
#   brittle-plan demo:
#   step 1: search(genai) ok
#   step 2 returned the unexpected -> REPLAN (1 extra planner call)

**Explanation**

A from-scratch Plan-and-Execute agent plus a second function that demonstrates the brittle-plan failure. Read it as: one planner call lays out every step as data, a dumb loop executes them, and when a step returns the unexpected you call the planner again.

**How the code works - step by step**
- **`plan(task)`** - ONE planner call that returns all five steps (two searches, two calcs, an answer) as a list before any tool runs.
- **`execute(task)`** - runs each step cheaply, collecting results, and assembles the final comparison at the `answer` step - no reasoning between steps.
- **`execute_with_replan(bad_step)`** - the failure demo: when step 2 returns something unexpected, it stops and reports a REPLAN (one extra planner call).
- The plan is a plain inspectable list - a real benefit over ReAct's hidden path.

**In one line:** one plan call up front + cheap execution; a surprise triggers a replan.

**What you'll see:** Prints the 5 planned steps executing in order (both prices looked up, both GST calcs, then the GenAI 17698.82 vs Python 11798.82 comparison), followed by the brittle-plan demo where step 2's surprise triggers a REPLAN message.

## 5 - ReWOO: reasoning without observation

ReWOO is Plan-and-Execute taken to its efficient extreme. The planner writes every step at once with placeholders (`#E1`, `#E2`) where results will land, without pausing to observe any of them; all tools then run in one batch, and a single solve call integrates the evidence. The result is exactly two model calls no matter how many tools - roughly 5x more token-efficient than ReAct - at the cost of zero mid-run adaptation.

In [ ]:
# ReWOO - Reasoning WithOut Observation. Plan ALL steps with #E placeholders, run tools in a
# batch, then solve once. Exactly TWO LLM calls (plan + solve), no matter how many tools.
KB = {"genai": "GenAI course: 14999 INR", "python": "Python course: 9999 INR"}
def search(q):  return KB.get(q.lower().split()[0], "not found")
def calc(expr): return round(eval(expr, {"__builtins__": {}}), 2)
TOOLS = {"search": search, "calc": calc}
calls = {"n": 0}

def planner():                                    # LLM call #1: plan with placeholders
    calls["n"] += 1
    return [("#E1", "calc", "14999*1.18"),        # steps do not wait on each other
            ("#E2", "calc", "9999*1.18")]

def solver(evidence):                             # LLM call #2: integrate the evidence
    calls["n"] += 1
    return f"GenAI {evidence['#E1']} vs Python {evidence['#E2']} INR, both incl. 18% GST"

def rewoo():
    plan = planner()
    evidence = {var: TOOLS[tool](arg) for var, tool, arg in plan}   # BATCH exec, parallelizable
    for var, tool, arg in plan:
        print(f"  {var} = {tool}({arg}) -> {evidence[var]}")
    print("  solve:", solver(evidence))
    print(f"  total LLM calls: {calls['n']}  (ReAct would need one PER step, growing with N)")

rewoo()

# Output:
#   #E1 = calc(14999*1.18) -> 17698.82
#   #E2 = calc(9999*1.18) -> 11798.82
#   solve: GenAI 17698.82 vs Python 11798.82 INR, both incl. 18% GST
#   total LLM calls: 2  (ReAct would need one PER step, growing with N)

**Explanation**

A from-scratch ReWOO agent that makes the two-call structure explicit with a call counter. Read it as: plan with blanks, fill every blank in a batch (independent tools are parallelizable), then solve once - the placeholder chain is what lets the tools run without waiting on each other.

**How the code works - step by step**
- **`planner()`** - LLM call #1: returns steps as `(placeholder, tool, arg)` tuples that do not depend on each other.
- **`solver(evidence)`** - LLM call #2: takes the filled-in evidence dict and writes the final answer.
- **`rewoo()`** - the driver: build the plan, fill every `#E` placeholder in a single dict comprehension (the batch, parallelizable), print each, then call `solver`.
- **`calls["n"]`** - proves the count stays at 2 regardless of tool count, versus ReAct's one-per-step.

**In one line:** plan with placeholders -> batch the tools -> solve once = exactly 2 calls.

**What you'll see:** Prints `#E1` and `#E2` resolving to the two GST-inclusive prices, the solver's combined GenAI-vs-Python sentence, and `total LLM calls: 2` with a note that ReAct would need one per step.

## 6 - Reflexion: generate, critique, retry

The earlier patterns decide how to reach an answer; Reflexion adds a quality loop around it. The agent generates a full attempt, an evaluator scores it and lists flaws, and if it falls short the agent writes a reflection ('I forgot the GST'), stores it in memory, and retries with that critique in context. It genuinely learns within a session - but each retry is a full run, making it the most expensive pattern, and it earns its keep only when there is a checkable quality bar.

In [ ]:
# REFLEXION - generate, self-critique, RETRY with the critique in memory. Quality via iteration.
def generate(task, reflections):                  # one full attempt (an LLM call)
    if not reflections:
        return "GenAI course is 14999 INR."                       # attempt 1: forgot the GST
    return "GenAI course is 14999 INR; with 18% GST that is 17698.82 INR."  # attempt 2: fixed

def evaluate(answer):                             # the evaluator scores + lists flaws
    if "GST" not in answer and "17698" not in answer:
        return {"score": 6, "flaws": ["did not apply the 18% GST the task asked for"]}
    return {"score": 9, "flaws": []}

def reflexion(task, max_attempts=3, bar=8):
    reflections = []
    for attempt in range(1, max_attempts + 1):
        answer = generate(task, reflections)
        verdict = evaluate(answer)
        print(f"  attempt {attempt}: score {verdict['score']}/10  ->  {answer}")
        if verdict["score"] >= bar:
            print(f"  accepted after {attempt} attempt(s)"); return
        reflections.append(f"Reflection: {verdict['flaws'][0]}")   # stored, fed to next attempt
        print(f"    reflect: {reflections[-1]}")

reflexion("Give the GenAI course price WITH 18% GST.")

# Output:
#   attempt 1: score 6/10  ->  GenAI course is 14999 INR.
#     reflect: Reflection: did not apply the 18% GST the task asked for
#   attempt 2: score 9/10  ->  GenAI course is 14999 INR; with 18% GST that is 17698.82 INR.
#   accepted after 2 attempt(s)

**Explanation**

A from-scratch Reflexion loop: generate, evaluate, reflect, retry. Read it as an outer quality loop wrapping the answer, where the stored reflection is what carries the lesson from one attempt into the next. Both `generate` and `evaluate` are scripted so the improvement is deterministic.

**How the code works - step by step**
- **`generate(task, reflections)`** - one full attempt (a model call); with no reflections it forgets the GST, with a reflection present it applies it.
- **`evaluate(answer)`** - the evaluator: scores 6/10 with a specific flaw if the GST is missing, else 9/10.
- **`reflexion(task, max_attempts, bar)`** - the loop: generate, score, accept if at/above the `bar` (8), else append the flaw as a reflection and retry.
- The stored reflection fed into attempt 2 is the mechanism - memory across attempts, not a smarter model.

**In one line:** generate -> self-critique -> retry with the critique in memory until it clears the bar.

**What you'll see:** Prints attempt 1 scoring 6/10 (forgot the GST) and its reflection, then attempt 2 scoring 9/10 with the GST applied, and 'accepted after 2 attempt(s)'.

## 7 - Routing: classify, then send to a specialized handler

The four patterns above are agent shapes where the model decides the path; the workflow patterns are the opposite - you fix the control flow in code, and they are often the highest-ROI choice. Routing is the standout: a cheap classifier reads the input, picks a category, and hands off to a specialized handler with its own prompt, tools, and model. Quality goes up and cost goes down; the one thing that can sink it is classification accuracy.

In [ ]:
# ROUTING - a cheap classifier sends each query to a SPECIALIZED handler (own model/prompt/tools).
# The highest-ROI workflow pattern when inputs fall into known categories.
def classify(query):                              # the router (a small, cheap call)
    q = query.lower()
    if any(w in q for w in ("refund", "charge", "invoice", "bill")): return "billing"
    if any(w in q for w in ("error", "crash", "bug", "install")):    return "technical"
    if any(w in q for w in ("return", "exchange", "replace")):       return "returns"
    return "general"

HANDLERS = {                                      # each handler is tuned for its domain
    "billing":   lambda q: "[billing/Haiku + account tools] checking your invoice...",
    "technical": lambda q: "[technical/Sonnet + docs RAG] reproducing the error...",
    "returns":   lambda q: "[returns/order tools] starting a return...",
    "general":   lambda q: "[general] routing to a human.",
}
def route(query):
    cat = classify(query)
    print(f"  {query!r:42s} -> {cat:9s} -> {HANDLERS[cat](query)}")

for q in ["I was charged twice this month",
          "the app crashes on install",
          "I want to return my order",
          "what are your opening hours"]:
    route(q)

# Output:
#   'I was charged twice this month'           -> billing   -> [billing/Haiku + account tools] checking your invoice...
#   'the app crashes on install'               -> technical -> [technical/Sonnet + docs RAG] reproducing the error...
#   'I want to return my order'                -> returns   -> [returns/order tools] starting a return...
#   'what are your opening hours'              -> general   -> [general] routing to a human.

**Explanation**

A from-scratch routing workflow: a keyword classifier plus a dispatch table of specialized handlers. Read it as fixed control flow you wrote, not an agent - the model would only live inside the classifier and the individual handlers, each tuned for its domain.

**How the code works - step by step**
- **`classify(query)`** - the cheap router: keyword matching maps each query to billing / technical / returns / general.
- **`HANDLERS`** - a dict of specialized handlers, each annotated with its intended model and tools (Haiku + account tools for billing, Sonnet + docs RAG for technical, etc.).
- **`route(query)`** - classifies, then dispatches to the matching handler and prints the hand-off.
- The loop runs four sample queries; a misroute would send one to the wrong specialist, which is why the classifier's accuracy caps the whole system.

**In one line:** a cheap classifier fans each input to a specialized handler; invest in the router.

**What you'll see:** Prints each of the four queries mapped to its category and handler - 'charged twice' -> billing, 'crashes on install' -> technical, 'return my order' -> returns, 'opening hours' -> general.

## 8 - The decision ladder: choose the simplest rung that fits

With the shapes in hand, the skill is choosing. Anthropic's guidance is a ladder: start at the simplest rung and climb only when a measurement shows the added complexity pays off - single call -> chaining -> routing -> parallelization -> Plan/ReWOO -> ReAct -> Reflexion -> orchestrator-workers -> multi-agent. The dominant anti-pattern is climbing for fashion; every added call and hop compounds cost and failure, so the burden of proof is on the more complex option.

In [ ]:
# THE DECISION LADDER - start at the simplest rung; climb ONLY when the task forces it.
def choose(single_shot, fixed_steps, known_categories, independent_parallel,
           path_unknown, decomposes_cleanly, checkable_quality):
    if single_shot:            return "just call the API",     "one shot, no control flow needed"
    if fixed_steps:            return "prompt chaining",       "fixed sub-steps with gates between"
    if known_categories:       return "routing",               "classify -> specialized handler"
    if independent_parallel:   return "parallelization",       "independent subtasks, or vote"
    if decomposes_cleanly:     return "Plan-and-Execute/ReWOO","plan up front, cheap execution"
    if path_unknown:           return "ReAct (agent)",         "decide the next step at runtime"
    if checkable_quality:      return "Reflexion",             "a clear quality bar to iterate against"
    return "orchestrator-workers -> multi-agent (11.5)", "dynamic subtasks; go up only if measured"

cases = [
    ("translate one line",       dict(single_shot=1, fixed_steps=0, known_categories=0, independent_parallel=0, path_unknown=0, decomposes_cleanly=0, checkable_quality=0)),
    ("summarize then translate", dict(single_shot=0, fixed_steps=1, known_categories=0, independent_parallel=0, path_unknown=0, decomposes_cleanly=0, checkable_quality=0)),
    ("support triage (3 cats)",  dict(single_shot=0, fixed_steps=0, known_categories=1, independent_parallel=0, path_unknown=0, decomposes_cleanly=0, checkable_quality=0)),
    ("draft code to a test",     dict(single_shot=0, fixed_steps=0, known_categories=0, independent_parallel=0, path_unknown=0, decomposes_cleanly=0, checkable_quality=1)),
    ("open-ended web research",  dict(single_shot=0, fixed_steps=0, known_categories=0, independent_parallel=0, path_unknown=1, decomposes_cleanly=0, checkable_quality=0)),
]
for name, flags in cases:
    pick, why = choose(**flags)
    print(f"  {name:26s} -> {pick:34s} ({why})")

# Output:
#   translate one line         -> just call the API                  (one shot, no control flow needed)
#   summarize then translate   -> prompt chaining                    (fixed sub-steps with gates between)
#   support triage (3 cats)    -> routing                            (classify -> specialized handler)
#   draft code to a test       -> Reflexion                          (a clear quality bar to iterate against)
#   open-ended web research    -> ReAct (agent)                      (decide the next step at runtime)

**Explanation**

An executable version of the decision ladder - a function that encodes the rungs as an ordered if-chain and returns the lowest rung whose condition a task meets. Read it as the lesson's judgment made runnable: task shape in, recommended architecture out.

**How the code works - step by step**
- **`choose(...)`** - takes boolean flags describing a task and returns the first (lowest) matching rung with a one-line reason; order encodes the ladder from single call up to multi-agent.
- **`cases`** - five example tasks, each described by which flags are set.
- **The loop** - runs `choose` on each and prints the task, the picked pattern, and why.
- The last rung (orchestrator-workers -> multi-agent) is deferred to Lesson 11.5 - climb there only when measured.

**In one line:** return the lowest rung the task's shape justifies, never higher.

**What you'll see:** Prints five tasks each settling on its lowest fitting rung: translate one line -> just call the API, summarize then translate -> prompt chaining, support triage -> routing, draft code to a test -> Reflexion, open-ended web research -> ReAct.

You built every canonical single-agent shape over one identical tool set and watched the wiring - not the model - move the numbers: ReAct spends one call per step for full adaptivity, Plan-and-Execute and ReWOO trade adaptivity for far fewer calls, Reflexion buys quality with repeated full runs, and routing plus the decision ladder keep you on the cheapest rung that clears the bar. The takeaway is the ladder: start simple and climb only when a measurement forces it, because every added call and hop compounds cost and failure. Next in Module 11, Lesson 11.3 gives these shapes a real state machine in LangGraph (checkpointing, human-in-the-loop), 11.4 replaces the toy history with real memory, and 11.5 moves from single-agent to multi-agent coordination.